# 01 — Creación del Dataset Maestro Granada

Objetivo: construir una primera versión del dataset maestro del proyecto **CoopScore Granada** a partir de datos GIS de suelo residencial y datos oficiales de población municipal.

Entradas esperadas:

- `data/raw/suelo_granada.csv`
- `data/raw/suelo_granada_clasificado.csv`
- `data/raw/suelo_granada_residencial.csv`
- `data/raw/2871.xlsx`

Salidas generadas:

- `data/processed/municipios_granada.csv`
- `data/processed/suelo_residencial_municipio.csv`
- `data/processed/dataset_maestro_granada_v1.csv`


In [19]:
# ============================================================
# 0. IMPORTACIÓN DE LIBRERÍAS Y CONFIGURACIÓN
# ============================================================

from pathlib import Path
import re
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')


# Si ejecutamos desde notebooks/
BASE_DIR = Path.cwd().parent

DATA_RAW = BASE_DIR / "data" / "raw"
DATA_PROCESSED = BASE_DIR / "data" / "processed"

print("BASE_DIR:", BASE_DIR)
print("DATA_RAW:", DATA_RAW)

BASE_DIR: c:\CoopScore_Granada
DATA_RAW: c:\CoopScore_Granada\data\raw


In [20]:
# ============================================================
# 1. CARGA DE DATOS RAW
# ============================================================

suelo = pd.read_csv(DATA_RAW / 'suelo_granada.csv')
suelo_clasificado = pd.read_csv(DATA_RAW / 'suelo_granada_clasificado.csv')
suelo_residencial = pd.read_csv(DATA_RAW / 'suelo_granada_residencial.csv')
ine_raw = pd.read_excel(DATA_RAW / '2871.xlsx', header=None)

print('suelo:', suelo.shape)
print('suelo_clasificado:', suelo_clasificado.shape)
print('suelo_residencial:', suelo_residencial.shape)
print('ine_raw:', ine_raw.shape)

suelo: (262, 5)
suelo_clasificado: (2000, 7)
suelo_residencial: (345, 7)
ine_raw: (191, 91)


c:\Users\pepetorres\AppData\Local\Programs\Python\Python314\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [21]:
# Vista rápida de los datos GIS residenciales
suelo_residencial.head()

,OBJECTID,codigo_ine,UGL_SG,NOMBRE,area_m2,SHAPE.STLength(),uso_suelo
0,1,18001,2.00,NaN,"62,405.08","1,185.38",Residencial
1,2,18002,2.00,NaN,"150,962.61","2,758.68",Residencial
2,5,18004,2.00,NaN,"213,613.19","1,948.35",Residencial
3,10,18006,2.00,NaN,"7,759.43",371.90,Residencial
4,13,18006,2.00,NaN,"10,586.52",492.23,Residencial


In [22]:
# Vista rápida del Excel INE
ine_raw.head(15)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90
0,Cifras oficiales de población de los municipio...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Municipal breakdown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Granada: Population by municipality and sex.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Unidades: Persons,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,,Total,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Males,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Females,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,,2025,"2,024.00","2,023.00","2,022.00","2,021.00","2,020.00","2,019.00",2018,2017,2016,2015,2014,2013,2012,2011,2010,2009,2008,2007,2006,2005,2004,2003,2002,2001,2000,1999,1998,1997,1996,2025,"2,024.00","2,023.00","2,022.00","2,021.00","2,020.00","2,019.00",2018,2017,2016,2015,2014,2013,2012,2011,2010,2009,2008,2007,2006,2005,2004,2003,2002,2001,2000,1999,1998,1997,1996,2025,"2,024.00","2,023.00","2,022.00","2,021.00","2,020.00","2,019.00",2018,2017,2016,2015,2014,2013,2012,2011,2010,2009,2008,2007,2006,2005,2004,2003,2002,2001,2000,1999,1998,1997,1996
8,18001 Agrón,247,244.00,239.00,247.00,255.00,260.00,274.00,288,290,297,310,313,324,342,330,340,358,381,374,283,301,320,344,354,380,392,415,425,,440,131,125.00,126.00,130.00,131.00,131.00,134.00,141,146,149,157,162,168,185,182,178,182,199,190,150,158,169,180,181,192,202,214,222,,228,116,119.00,113.00,117.00,124.00,129.00,140.00,147,144,148,153,151,156,157,148,162,176,182,184,133,143,151,164,173,188,190,201,203,,212
9,18002 Alamedilla,518,543.00,562.00,567.00,564.00,569.00,574.00,599,606

In [23]:
# ============================================================
# 2. LIMPIEZA DEL EXCEL INE: CÓDIGO INE, MUNICIPIO Y POBLACIÓN
# ============================================================

# En el Excel, a partir de la fila 8 aparecen los municipios.
# Columna 0: '18001 Agrón'
# Columna 1: población total 2025

ine = ine_raw.iloc[8:, [0, 1]].copy()
ine.columns = ['municipio_raw', 'poblacion_2025']

# Eliminamos filas vacías o no municipales
ine = ine.dropna(subset=['municipio_raw'])
ine['municipio_raw'] = ine['municipio_raw'].astype(str).str.strip()

# Nos quedamos solo con filas que empiezan por 5 dígitos INE
ine = ine[ine['municipio_raw'].str.match(r'^\d{5}\s+')].copy()

# Extraemos código y nombre
ine['codigo_ine'] = ine['municipio_raw'].str.extract(r'^(\d{5})')
ine['municipio'] = ine['municipio_raw'].str.replace(r'^\d{5}\s+', '', regex=True).str.strip()

# Convertimos población a numérico
ine['poblacion_2025'] = (
    ine['poblacion_2025']
    .astype(str)
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
)
ine['poblacion_2025'] = pd.to_numeric(ine['poblacion_2025'], errors='coerce')

municipios = ine[['codigo_ine', 'municipio', 'poblacion_2025']].copy()
municipios.head()

,codigo_ine,municipio,poblacion_2025
8,18001,Agrón,247
9,18002,Alamedilla,518
10,18003,Albolote,19768
11,18004,Albondón,703
12,18006,Albuñol,7388


In [24]:
# Comprobación básica de municipios
print('Municipios INE cargados:', municipios.shape[0])
print('Duplicados codigo_ine:', municipios['codigo_ine'].duplicated().sum())
municipios.sample(10, random_state=42)

Municipios INE cargados: 174
Duplicados codigo_ine: 0


,codigo_ine,municipio,poblacion_2025
164,18914,Valderrubio,2069
153,18174,Santa Cruz del Comercio,523
110,18121,Lobras,139
136,18152,Pedro Martínez,1166
149,18168,Quéntar,952
147,18159,Píñar,1040
50,18048,Cijuela,3827
23,18904,Alpujarra de la Sierra,933
135,18151,Pampaneira,312
73,18072,Escúzar,826


In [25]:
# ============================================================
# 3. LIMPIEZA Y RESUMEN DEL SUELO RESIDENCIAL
# ============================================================

res = suelo_residencial.copy()

# Normalizamos código INE a string de 5 dígitos
res['codigo_ine'] = res['codigo_ine'].astype(int).astype(str).str.zfill(5)

# Renombramos columnas técnicas
res = res.rename(columns={
    'SHAPE.STLength()': 'perimetro_m',
    'area_m2': 'superficie_poligono_m2'
})

# Aseguramos tipos numéricos
res['superficie_poligono_m2'] = pd.to_numeric(res['superficie_poligono_m2'], errors='coerce')
res['perimetro_m'] = pd.to_numeric(res['perimetro_m'], errors='coerce')

# Validaciones rápidas
print('Registros residenciales:', len(res))
print('Municipios con suelo residencial:', res['codigo_ine'].nunique())
print('Área total residencial m²:', round(res['superficie_poligono_m2'].sum(), 2))

res.head()

Registros residenciales: 345
Municipios con suelo residencial: 128
Área total residencial m²: 84520529.52


,OBJECTID,codigo_ine,UGL_SG,NOMBRE,superficie_poligono_m2,perimetro_m,uso_suelo
0,1,18001,2.00,NaN,"62,405.08","1,185.38",Residencial
1,2,18002,2.00,NaN,"150,962.61","2,758.68",Residencial
2,5,18004,2.00,NaN,"213,613.19","1,948.35",Residencial
3,10,18006,2.00,NaN,"7,759.43",371.90,Residencial
4,13,18006,2.00,NaN,"10,586.52",492.23,Residencial


In [26]:
# Agrupamos por municipio
suelo_resumen = (
    res
    .groupby('codigo_ine')
    .agg(
        n_poligonos_residenciales=('OBJECTID', 'count'),
        superficie_residencial_m2=('superficie_poligono_m2', 'sum'),
        superficie_media_poligono_m2=('superficie_poligono_m2', 'mean'),
        superficie_mediana_poligono_m2=('superficie_poligono_m2', 'median'),
        superficie_max_poligono_m2=('superficie_poligono_m2', 'max'),
        perimetro_total_m=('perimetro_m', 'sum')
    )
    .reset_index()
)

suelo_resumen['superficie_residencial_ha'] = suelo_resumen['superficie_residencial_m2'] / 10_000

suelo_resumen = suelo_resumen.sort_values('superficie_residencial_m2', ascending=False)
suelo_resumen.head(15)

,codigo_ine,n_poligonos_residenciales,superficie_residencial_m2,superficie_media_poligono_m2,superficie_mediana_poligono_m2,superficie_max_poligono_m2,perimetro_total_m,superficie_residencial_ha
120,18905,1,"3,682,034.80","3,682,034.80","3,682,034.80","3,682,034.80","40,983.79",368.20
60,18089,7,"3,460,927.70","494,418.24","118,598.31","2,848,795.58","43,493.54",346.09
16,18022,1,"3,364,039.55","3,364,039.55","3,364,039.55","3,364,039.55","37,294.92",336.40
21,18029,9,"3,295,178.29","366,130.92","108,955.23","1,376,322.69","32,558.92",329.52
2,18003,1,"2,949,835.53","2,949,835.53","2,949,835.53","2,949,835.53","42,092.59",294.98
78,18122,19,"2,865,887.93","150,836.21","59,364.89","1,489,210.99","54,607.58",286.59
106,18173,1,"2,549,806.21","2,549,806.21","2,549,806.21","2,549,806.21","37,748.82",254.98
66,18102,9,"2,251,831.88","250,203.54","155,067.85","826,560.63","41,696.83",225.18
17,18023,7,"2,195,843.08","313,691.87","40,544.71","1,948,584.38","42,292.24",219.58
15,18021,3,"2,018,746.32","672,915.44","122,760.38","1,777,167.56","31,524.40",201.87


In [27]:
# ============================================================
# 4. CRUCE SUELO RESIDENCIAL + MUNICIPIOS INE
# ============================================================

dataset = suelo_resumen.merge(
    municipios,
    on='codigo_ine',
    how='left',
    validate='many_to_one'
)

# Reordenamos columnas
cols = [
    'codigo_ine', 'municipio', 'poblacion_2025',
    'n_poligonos_residenciales',
    'superficie_residencial_m2', 'superficie_residencial_ha',
    'superficie_media_poligono_m2', 'superficie_mediana_poligono_m2',
    'superficie_max_poligono_m2', 'perimetro_total_m'
]
dataset = dataset[cols]

# Variables derivadas útiles para EDA
# m² de suelo residencial por habitante
# cuidado: si población es 0 o nula, evitamos división inválida
dataset['suelo_residencial_m2_por_hab'] = np.where(
    dataset['poblacion_2025'] > 0,
    dataset['superficie_residencial_m2'] / dataset['poblacion_2025'],
    np.nan
)

# Orden final
dataset = dataset.sort_values('superficie_residencial_m2', ascending=False).reset_index(drop=True)

dataset.head(20)

,codigo_ine,municipio,poblacion_2025,n_poligonos_residenciales,superficie_residencial_m2,superficie_residencial_ha,superficie_media_poligono_m2,superficie_mediana_poligono_m2,superficie_max_poligono_m2,perimetro_total_m,suelo_residencial_m2_por_hab
0,18905,"Gabias, Las",23584,1,"3,682,034.80",368.20,"3,682,034.80","3,682,034.80","3,682,034.80","40,983.79",156.12
1,18089,Guadix,18881,7,"3,460,927.70",346.09,"494,418.24","118,598.31","2,848,795.58","43,493.54",183.30
2,18022,Atarfe,20914,1,"3,364,039.55",336.40,"3,364,039.55","3,364,039.55","3,364,039.55","37,294.92",160.85
3,18029,Benamaurel,2235,9,"3,295,178.29",329.52,"366,130.92","108,955.23","1,376,322.69","32,558.92","1,474.35"
4,18003,Albolote,19768,1,"2,949,835.53",294.98,"2,949,835.53","2,949,835.53","2,949,835.53","42,092.59",149.22
5,18122,Loja,20951,19,"2,865,887.93",286.59,"150,836.21","59,364.89","1,489,210.99","54,607.58",136.79
6,18173,Salobreña,12760,1,"2,549,806.21",254.98,"2,549,806.21","2,549,806.21","2,549,806.21","37,748.82",199.83
7,18102,Íllora,9923,9,"2,251,831.88",225.18,"250,203.54","155,067.85","826,560.63","41,696.83",226.93
8,18023,Baza,20587,7,"2,195,843.08",219.58,"313,691.87","40,544.71","1,948,584.38","42,292.24",106.66
9,18021,Armilla,25300,3,"2,018,746.32",201.87,"672,915.44","122,760.38","1,777,167.56","31,524.40",79.79


In [28]:
# ==========================================================
# RECONSTRUCCIÓN COMPLETA DEL DATASET MAESTRO
# ==========================================================

# 1. Resumen de suelo residencial por municipio
resumen_suelo = (
    suelo_residencial
    .groupby("codigo_ine")
    .agg(
        superficie_residencial_m2=("area_m2", "sum"),
        n_poligonos=("OBJECTID", "count"),
        superficie_media_poligono=("area_m2", "mean")
    )
    .reset_index()
)

# 2. Municipios desde Excel INE
municipios = ine_raw.iloc[8:, [0, 1]].copy()
municipios.columns = ["municipio_raw", "poblacion_2025"]
municipios = municipios.dropna()

municipios["codigo_ine"] = (
    municipios["municipio_raw"]
    .astype(str)
    .str.extract(r"(\d{5})")
)

municipios["municipio"] = (
    municipios["municipio_raw"]
    .astype(str)
    .str.replace(r"^\d{5}\s*", "", regex=True)
    .str.strip()
)

municipios = municipios.dropna(subset=["codigo_ine"])
municipios["codigo_ine"] = municipios["codigo_ine"].astype(int)

municipios = municipios[
    ["codigo_ine", "municipio", "poblacion_2025"]
]

# 3. Merge
dataset_maestro = resumen_suelo.merge(
    municipios,
    on="codigo_ine",
    how="left"
)

# 4. Orden final
dataset_maestro = dataset_maestro[
    [
        "codigo_ine",
        "municipio",
        "poblacion_2025",
        "superficie_residencial_m2",
        "n_poligonos",
        "superficie_media_poligono"
    ]
].sort_values("superficie_residencial_m2", ascending=False)

print("resumen_suelo:", resumen_suelo.shape)
print("municipios:", municipios.shape)
print("dataset_maestro:", dataset_maestro.shape)

display(dataset_maestro.head(20))

resumen_suelo: (128, 4)
municipios: (174, 3)
dataset_maestro: (128, 6)


,codigo_ine,municipio,poblacion_2025,superficie_residencial_m2,n_poligonos,superficie_media_poligono
120,18905,"Gabias, Las",23584,"3,682,034.80",1,"3,682,034.80"
60,18089,Guadix,18881,"3,460,927.70",7,"494,418.24"
16,18022,Atarfe,20914,"3,364,039.55",1,"3,364,039.55"
21,18029,Benamaurel,2235,"3,295,178.29",9,"366,130.92"
2,18003,Albolote,19768,"2,949,835.53",1,"2,949,835.53"
78,18122,Loja,20951,"2,865,887.93",19,"150,836.21"
106,18173,Salobreña,12760,"2,549,806.21",1,"2,549,806.21"
66,18102,Íllora,9923,"2,251,831.88",9,"250,203.54"
17,18023,Baza,20587,"2,195,843.08",7,"313,691.87"
15,18021,Armilla,25300,"2,018,746.32",3,"672,915.44"


In [29]:
# ============================================================
# 5. CONTROL DE CALIDAD DEL CRUCE
# ============================================================

sin_nombre = dataset[dataset['municipio'].isna()]
print('Municipios sin nombre tras cruce:', len(sin_nombre))

if len(sin_nombre) > 0:
    display(sin_nombre[['codigo_ine', 'superficie_residencial_m2', 'n_poligonos_residenciales']])
else:
    print('Cruce correcto: todos los códigos INE del suelo residencial tienen municipio asociado.')

Municipios sin nombre tras cruce: 0
Cruce correcto: todos los códigos INE del suelo residencial tienen municipio asociado.


In [30]:
# Municipios clave para la demo
municipios_demo = [
    'Granada', 'Motril', 'Salobreña', 'Almuñécar',
    'Armilla', 'Maracena', 'Albolote', 'Atarfe',
    'Las Gabias', 'Ogíjares', 'Loja', 'Guadix', 'Baza'
]

# Búsqueda flexible por si hay tildes o nombres compuestos diferentes
patron = '|'.join(municipios_demo)

demo = dataset[dataset['municipio'].fillna('').str.contains(patron, case=False, regex=True)].copy()
demo.sort_values('superficie_residencial_m2', ascending=False)

,codigo_ine,municipio,poblacion_2025,n_poligonos_residenciales,superficie_residencial_m2,superficie_residencial_ha,superficie_media_poligono_m2,superficie_mediana_poligono_m2,superficie_max_poligono_m2,perimetro_total_m,suelo_residencial_m2_por_hab
1,18089,Guadix,18881,7,"3,460,927.70",346.09,"494,418.24","118,598.31","2,848,795.58","43,493.54",183.30
2,18022,Atarfe,20914,1,"3,364,039.55",336.40,"3,364,039.55","3,364,039.55","3,364,039.55","37,294.92",160.85
4,18003,Albolote,19768,1,"2,949,835.53",294.98,"2,949,835.53","2,949,835.53","2,949,835.53","42,092.59",149.22
5,18122,Loja,20951,19,"2,865,887.93",286.59,"150,836.21","59,364.89","1,489,210.99","54,607.58",136.79
6,18173,Salobreña,12760,1,"2,549,806.21",254.98,"2,549,806.21","2,549,806.21","2,549,806.21","37,748.82",199.83
8,18023,Baza,20587,7,"2,195,843.08",219.58,"313,691.87","40,544.71","1,948,584.38","42,292.24",106.66
9,18021,Armilla,25300,3,"2,018,746.32",201.87,"672,915.44","122,760.38","1,777,167.56","31,524.40",79.79
12,18017,Almuñécar,27544,14,"1,697,002.90",169.70,"121,214.49","70,834.09","434,641.19","39,058.33",61.61
19,18127,Maracena,22294,4,"1,233,802.70",123.38,"308,450.67","31,368.64","1,169,108.11","23,592.45",55.34
25,18053,Cortes de Baza,1769,8,"959,226.33",95.92,"119,903.29","100,042.67","383,662.37","25,198.71",542.24


In [31]:
# ============================================================
# 6. EXPORTACIÓN DE RESULTADOS
# ============================================================

municipios.to_csv(DATA_PROCESSED / 'municipios_granada.csv', index=False, encoding='utf-8-sig')
suelo_resumen.to_csv(DATA_PROCESSED / 'suelo_residencial_municipio.csv', index=False, encoding='utf-8-sig')
dataset.to_csv(DATA_PROCESSED / 'dataset_maestro_granada_v1.csv', index=False, encoding='utf-8-sig')

print('Archivos generados:')
print('-', DATA_PROCESSED / 'municipios_granada.csv')
print('-', DATA_PROCESSED / 'suelo_residencial_municipio.csv')
print('-', DATA_PROCESSED / 'dataset_maestro_granada_v1.csv')

Archivos generados:
- c:\CoopScore_Granada\data\processed\municipios_granada.csv
- c:\CoopScore_Granada\data\processed\suelo_residencial_municipio.csv
- c:\CoopScore_Granada\data\processed\dataset_maestro_granada_v1.csv


In [32]:
# ============================================================
# 7. RESUMEN EJECUTIVO PARA DOCUMENTACIÓN
# ============================================================

resumen = {
    'municipios_ine_total': municipios.shape[0],
    'poligonos_residenciales_total': res.shape[0],
    'municipios_con_suelo_residencial': dataset.shape[0],
    'superficie_residencial_total_m2': round(dataset['superficie_residencial_m2'].sum(), 2),
    'superficie_residencial_total_ha': round(dataset['superficie_residencial_ha'].sum(), 2),
    'poblacion_total_municipios_con_suelo': int(dataset['poblacion_2025'].sum())
}

resumen

{'municipios_ine_total': 174,
 'poligonos_residenciales_total': 345,
 'municipios_con_suelo_residencial': 128,
 'superficie_residencial_total_m2': np.float64(84520529.52),
 'superficie_residencial_total_ha': np.float64(8452.05),
 'poblacion_total_municipios_con_suelo': 558309}

In [33]:
print("Variables disponibles:")
print([v for v in globals().keys() if not v.startswith("_")])

print("\n¿Existe resumen_suelo?", "resumen_suelo" in globals())
print("¿Existe municipios?", "municipios" in globals())
print("¿Existe dataset_maestro?", "dataset_maestro" in globals())

Variables disponibles:
['In', 'Out', 'get_ipython', 'exit', 'quit', 'open', 'Path', 're', 'np', 'pd', 'BASE_DIR', 'DATA_RAW', 'DATA_PROCESSED', 'suelo', 'suelo_clasificado', 'suelo_residencial', 'ine_raw', 'ine', 'municipios', 'res', 'suelo_resumen', 'dataset', 'cols', 'sin_nombre', 'municipios_demo', 'patron', 'demo', 'resumen', 'resumen_suelo', 'dataset_maestro']

¿Existe resumen_suelo? True
¿Existe municipios? True
¿Existe dataset_maestro? True


In [34]:
# ==========================================================
# CREACIÓN DEL DATASET MAESTRO
# ==========================================================

print("Columnas resumen_suelo:")
print(resumen_suelo.columns)

print("\nColumnas municipios:")
print(municipios.columns)

# Aseguramos mismo tipo de dato
resumen_suelo["codigo_ine"] = resumen_suelo["codigo_ine"].astype(int)
municipios["codigo_ine"] = municipios["codigo_ine"].astype(int)

dataset_maestro = resumen_suelo.merge(
    municipios,
    on="codigo_ine",
    how="left"
)

print("Dataset maestro creado:", dataset_maestro.shape)

dataset_maestro.head()

Columnas resumen_suelo:
Index(['codigo_ine', 'superficie_residencial_m2', 'n_poligonos',
       'superficie_media_poligono'],
      dtype='object')

Columnas municipios:
Index(['codigo_ine', 'municipio', 'poblacion_2025'], dtype='object')
Dataset maestro creado: (128, 6)


,codigo_ine,superficie_residencial_m2,n_poligonos,superficie_media_poligono,municipio,poblacion_2025
0,18001,"62,405.08",1,"62,405.08",Agrón,247
1,18002,"150,962.61",1,"150,962.61",Alamedilla,518
2,18003,"2,949,835.53",1,"2,949,835.53",Albolote,19768
3,18004,"213,613.19",1,"213,613.19",Albondón,703
4,18006,"575,638.39",10,"57,563.84",Albuñol,7388


In [35]:
# ==========================================================
# VALIDACIÓN DEL CRUCE
# ==========================================================

print("Filas dataset maestro:", dataset_maestro.shape[0])
print("Columnas:", dataset_maestro.shape[1])

print("Municipios sin nombre tras el cruce:")
display(dataset_maestro[dataset_maestro["municipio"].isna()])

dataset_maestro.head(10)

Filas dataset maestro: 128
Columnas: 6
Municipios sin nombre tras el cruce:


,codigo_ine,superficie_residencial_m2,n_poligonos,superficie_media_poligono,municipio,poblacion_2025


,codigo_ine,superficie_residencial_m2,n_poligonos,superficie_media_poligono,municipio,poblacion_2025
0,18001,"62,405.08",1,"62,405.08",Agrón,247
1,18002,"150,962.61",1,"150,962.61",Alamedilla,518
2,18003,"2,949,835.53",1,"2,949,835.53",Albolote,19768
3,18004,"213,613.19",1,"213,613.19",Albondón,703
4,18006,"575,638.39",10,"57,563.84",Albuñol,7388
5,18007,"167,357.00",2,"83,678.50",Albuñuelas,794
6,18010,"211,494.48",2,"105,747.24",Aldeire,597
7,18011,"1,168,742.81",1,"1,168,742.81",Alfacar,5834
8,18012,"338,713.85",3,"112,904.62",Algarinejo,2335
9,18013,"861,031.63",5,"172,206.33",Alhama de Granada,5544


In [36]:
# ==========================================================
# ORDEN Y COLUMNAS FINALES
# ==========================================================

dataset_maestro = dataset_maestro[
    [
        "codigo_ine",
        "municipio",
        "poblacion_2025",
        "superficie_residencial_m2",
        "n_poligonos",
        "superficie_media_poligono"
    ]
].sort_values("superficie_residencial_m2", ascending=False)

dataset_maestro.head(20)

,codigo_ine,municipio,poblacion_2025,superficie_residencial_m2,n_poligonos,superficie_media_poligono
120,18905,"Gabias, Las",23584,"3,682,034.80",1,"3,682,034.80"
60,18089,Guadix,18881,"3,460,927.70",7,"494,418.24"
16,18022,Atarfe,20914,"3,364,039.55",1,"3,364,039.55"
21,18029,Benamaurel,2235,"3,295,178.29",9,"366,130.92"
2,18003,Albolote,19768,"2,949,835.53",1,"2,949,835.53"
78,18122,Loja,20951,"2,865,887.93",19,"150,836.21"
106,18173,Salobreña,12760,"2,549,806.21",1,"2,549,806.21"
66,18102,Íllora,9923,"2,251,831.88",9,"250,203.54"
17,18023,Baza,20587,"2,195,843.08",7,"313,691.87"
15,18021,Armilla,25300,"2,018,746.32",3,"672,915.44"


In [37]:
# ==========================================================
# GUARDADO DATASET MAESTRO V1
# ==========================================================

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

municipios.to_csv(DATA_PROCESSED / "municipios_granada.csv", index=False)
resumen_suelo.to_csv(DATA_PROCESSED / "suelo_residencial_municipio.csv", index=False)
dataset_maestro.to_csv(DATA_PROCESSED / "dataset_maestro_granada_v1.csv", index=False)

print("Archivos guardados correctamente en:", DATA_PROCESSED)

Archivos guardados correctamente en: c:\CoopScore_Granada\data\processed


In [38]:
import os

os.listdir(DATA_PROCESSED)

['dataset_maestro_granada_v1.csv',
 'municipios_granada.csv',
 'suelo_residencial_municipio.csv']

## Siguiente paso

Una vez generado `dataset_maestro_granada_v1.csv`, el siguiente notebook será:

`02_eda_granada.ipynb`

Ahí analizaremos:

- Distribución de superficie residencial por municipio.
- Ranking de municipios.
- Relación entre población y suelo residencial.
- Municipios con mayor suelo residencial por habitante.
- Selección de municipios prioritarios para la demo.


In [39]:
# ==========================================================
# CARGA PRECIOS VIVIENDA
# ==========================================================

precios = pd.read_csv(
    BASE_DIR / "data" / "external" / "precios_vivienda_granada.csv"
)

print(precios.shape)

precios.head()

(9, 4)


,municipio,precio_m2,fuente,anio
0,Granada,2628,Idealista,2026
1,Almuñécar,2999,Idealista,2026
2,Salobreña,2078,Idealista,2026
3,Motril,1491,Idealista,2026
4,Armilla,2120,Idealista,2026


In [40]:
# ==========================================================
# MERGE PRECIOS + DATASET MAESTRO
# ==========================================================

dataset_maestro_v2 = dataset_maestro.merge(
    precios,
    on="municipio",
    how="left"
)

print(dataset_maestro_v2.shape)

dataset_maestro_v2.head()

(128, 9)


,codigo_ine,municipio,poblacion_2025,superficie_residencial_m2,n_poligonos,superficie_media_poligono,precio_m2,fuente,anio
0,18905,"Gabias, Las",23584,"3,682,034.80",1,"3,682,034.80",NaN,NaN,NaN
1,18089,Guadix,18881,"3,460,927.70",7,"494,418.24",NaN,NaN,NaN
2,18022,Atarfe,20914,"3,364,039.55",1,"3,364,039.55","1,369.00",Idealista,"2,026.00"
3,18029,Benamaurel,2235,"3,295,178.29",9,"366,130.92",NaN,NaN,NaN
4,18003,Albolote,19768,"2,949,835.53",1,"2,949,835.53",NaN,NaN,NaN


In [41]:
# ==========================================================
# COBERTURA DE PRECIOS
# ==========================================================

print(
    "Municipios con precio:",
    dataset_maestro_v2["precio_m2"].notna().sum()
)

print(
    "Municipios sin precio:",
    dataset_maestro_v2["precio_m2"].isna().sum()
)

Municipios con precio: 7
Municipios sin precio: 121


In [42]:
# ==========================================================
# GUARDAR DATASET MAESTRO V2
# ==========================================================

dataset_maestro_v2.to_csv(
    DATA_PROCESSED / "dataset_maestro_granada_v2.csv",
    index=False
)

print("Dataset Maestro V2 guardado")

Dataset Maestro V2 guardado
